# Prompt-wording sensitivity pilot (Google Colab)

This notebook runs the existing reproducible **10-question × 5-prompt** pilot. It does not implement experiment logic itself: all sampling, prompting, Qwen inference, extraction, evaluation, and analysis come from the repository modules.

Before continuing, choose **Runtime → Change runtime type → T4 GPU**. Do not run `--full` in this notebook.

In [ ]:
# Replace with your repository's HTTPS clone URL.
REPO_URL = "https://github.com/<OWNER>/<REPOSITORY>.git"
REPO_DIR = "llm-prompt-sensitivity"

!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git rev-parse --short HEAD

In [ ]:
# Colab supplies CUDA-enabled PyTorch; install the remaining pinned project requirements.
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

import torch
assert torch.cuda.is_available(), "CUDA is unavailable. Select a T4 GPU runtime, then reconnect."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Local tests only: no model or dataset download is triggered here.
!python -m pytest -q

In [ ]:
# This invokes the repository runner with its centralized configuration:
# Qwen/Qwen3-1.7B, validation split, seed 42, enable_thinking=False,
# greedy decoding, max_new_tokens=16, and 10 questions × 5 variants.
!PYTHONPATH=. python -m src.runner --pilot

In [ ]:
# Analyze only the records just produced by the existing runner.
!PYTHONPATH=. python scripts/analyze.py outputs/pilot/generations.jsonl
!PYTHONPATH=. python scripts/plot_results.py outputs/pilot/generations.jsonl

from pathlib import Path
assert len(Path("outputs/pilot/generations.jsonl").read_text().splitlines()) == 50, "Expected exactly 50 pilot generations."
print("Verified: 50 pilot records.")

In [ ]:
# Package the raw JSONL, metadata, sampled IDs, summaries, and plot for download.
import shutil
from google.colab import files

archive = shutil.make_archive("outputs/pilot_artifacts", "zip", root_dir="outputs", base_dir="pilot")
files.download(archive)